# SQL Project: Exploring Movie Ratings with Databricks
Dataset: MovieLens. `u.data` contains user ratings, `u.item` contains movie information (title, release date).

Steps: 1. Load the data into tables · 2. Check the tables · 3. Part 1 · 4. Part 2


## 1. Load the data into tables

In [0]:
-- Load u.data (user ratings) into a table.
-- The file is tab-separated and has no header so the schema is defined explicitly.
CREATE OR REPLACE TABLE workspace.default.ratings AS
SELECT * FROM read_files(
  '/Volumes/movielens/default/movielens/u.data',
  format => 'csv', sep => '\t', header => false,
  schema => 'user_id INT, movie_id INT, rating INT, rating_ts BIGINT'
);

num_affected_rows,num_inserted_rows


In [0]:
-- Load u.item (movie information) into a table.
-- The file is pipe-separated (|) and encoded in ISO-8859-1 (accented titles).
-- Only the 3 useful columns are read: id, title and release date (as text).
-- try_to_date converts the text (e.g. 4-Feb-1971) into a real DATE so it can be sorted chronologically.
CREATE OR REPLACE TABLE workspace.default.movies AS
SELECT movie_id, title, try_to_date(release_date_str, 'd-MMM-yyyy') AS release_date
FROM read_files(
  '/Volumes/movielens/default/movielens/u.item',
  format => 'csv', sep => '|', header => false, encoding => 'ISO-8859-1',
  schema => 'movie_id INT, title STRING, release_date_str STRING'
);

num_affected_rows,num_inserted_rows


## 2. Check the tables

In [0]:
-- Count the rows of each table and the movies with a missing release date
SELECT 'ratings' AS table_name, COUNT(*) AS n FROM workspace.default.ratings
UNION ALL
SELECT 'movies', COUNT(*) FROM workspace.default.movies
UNION ALL
SELECT 'movies sans date', COUNT(*) FROM workspace.default.movies WHERE release_date IS NULL;

table_name,n
movies sans date,1
ratings,100000
movies,1682


## Data check
Quick check of the two tables before running the project queries.
Expected: 100,000 ratings, 1,682 movies, and 1 movie without a release date (the movie named "unknown").

## 3. Part 1: Find the Oldest Highly Rated Movies
**Question:** Which are the five oldest movies with an average rating greater than 4.0?

**Approach:** join the ratings to the movies on `movie_id`, compute the average rating per movie, keep the movies above 4.0, sort by release date and keep the first five.

In [0]:
SELECT m.movie_id, m.title, m.release_date,
       ROUND(AVG(r.rating), 2) AS avg_rating, -- average of the individual ratings from u.data (2 digits after the comma)
       COUNT(*) AS nb_ratings                 -- number of ratings behind each average
FROM workspace.default.ratings r
JOIN workspace.default.movies m ON r.movie_id = m.movie_id  -- link both datasets with the movie identifier
WHERE m.release_date IS NOT NULL                            -- exclude the "unknown" movie (no date)
GROUP BY m.movie_id, m.title, m.release_date                -- one row per movie
HAVING AVG(r.rating) > 4.0                                  -- filter on the average, after the grouping
ORDER BY m.release_date, m.movie_id
LIMIT 5;                                                    -- keep the five oldest

movie_id,title,release_date,avg_rating,nb_ratings
493,"Thin Man, The (1934)",1934-01-01,4.15,60
604,It Happened One Night (1934),1934-01-01,4.01,81
615,"39 Steps, The (1935)",1935-01-01,4.05,59
1203,Top Hat (1935),1935-01-01,4.05,21
613,My Man Godfrey (1936),1936-01-01,4.04,27


## 4. Part 2: The Most Popular Bad Movies
**Question:** Which poorly rated movies are the most popular among users?

**Approach:** compute the average rating and the number of ratings per movie, keep the movies below 2.0 and sort by number of ratings (popularity) from highest to lowest.

In [0]:
SELECT m.movie_id, m.title, m.release_date,
       ROUND(AVG(r.rating), 2) AS avg_rating,           -- average of the individual ratings
       COUNT(*) AS nb_ratings                           -- total number of ratings = popularity
FROM workspace.default.ratings r
JOIN workspace.default.movies m ON r.movie_id = m.movie_id      -- link both datasets with the movie identifier
GROUP BY m.movie_id, m.title, m.release_date                    -- one row per movie
HAVING AVG(r.rating) < 2.0                                      -- keep only poorly rated movies
ORDER BY nb_ratings DESC;                                       -- most rated first

movie_id,title,release_date,avg_rating,nb_ratings
688,Leave It to Beaver (1997),1997-08-22,1.84,44
890,Mortal Kombat: Annihilation (1997),1997-01-01,1.95,43
743,"Crow: City of Angels, The (1996)",1996-08-30,1.95,39
368,Bio-Dome (1996),1996-01-01,1.9,31
1215,Barb Wire (1996),1996-05-03,1.93,30
457,Free Willy 3: The Rescue (1997),1997-08-08,1.74,27
375,Showgirls (1995),1995-01-01,1.96,23
758,Lawnmower Man 2: Beyond Cyberspace (1996),1996-01-01,1.71,21
894,Home Alone 3 (1997),1997-01-01,1.89,19
424,Children of the Corn: The Gathering (1996),1996-01-01,1.32,19
